In [3]:
import torch
from torch import nn
from torch.nn import functional as F
from torch import optim
import dlc_practical_prologue as prologue
import time

In [5]:
# Compute samples for train, validation and test sets. Each set has N random samples 
N = 1000
input, target, classes, _, _, _= prologue.generate_pair_sets(3*N)
indices = torch.randperm(3*N)
train_input, train_target, train_classes = input[indices[:N]], target[indices[:N]], classes[indices[:N]]
val_input, val_target, val_classes = input[indices[N:2*N]], target[indices[N:2*N]], classes[indices[N:2*N]]
test_input, test_target, test_classes = input[indices[2*N:3*N]], target[indices[2*N:3*N]], classes[indices[2*N:3*N]]

In [10]:
# good, but a lot of overfitting (train accuracy of 1 after 25 epochs)

# version lugeon
class NetTestConv(nn.Module):
    def __init__(self):
        super(NetTest, self).__init__()
        self.conv1 = nn.Conv2d(2, 64, kernel_size=5)
        self.conv2 = nn.Conv2d(64, 64, kernel_size=2, stride=2)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=2)
        self.conv4 = nn.Conv2d(128, 128, kernel_size=3, stride=1)
        self.fc1 = nn.Linear(512, 20)
        self.fc2 = nn.Linear(20, 2)


    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = F.relu(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = F.relu(x)
        x = self.fc1(x.view(-1, 512))
        x = F.relu(x)
        x = self.fc2(x)
        return x

# version lugeon
class NetTest(nn.Module):
    def __init__(self):
        super(NetTest, self).__init__()
        self.conv1 = nn.Conv2d(2, 64, kernel_size=5)
        self.max_pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=2)
        self.max_pool2 = nn.MaxPool2d(kernel_size=3, stride=1)
        self.fc1 = nn.Linear(512, 20)
        self.fc2 = nn.Linear(20, 2)


    def forward(self, x):
        x = self.conv1(x)
        x = self.max_pool1(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = self.max_pool2(x)
        x = F.relu(x)
        x = self.fc1(x.view(-1, 512))
        x = F.relu(x)
        x = self.fc2(x)
        return x

# version lugeon
class NetTestDropout(nn.Module):
    def __init__(self):
        super(NetTestDropout, self).__init__()
        self.conv1 = nn.Conv2d(2, 64, kernel_size=5)
        self.max_pool1 = nn.MaxPool2d(64, 64, kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=2)
        self.max_pool2 = nn.MaxPool2d(128, 128, kernel_size=3, stride=1)
        self.fc1 = nn.Linear(512, 20)
        self.fc2 = nn.Linear(20, 2)
        self.drop2d = nn.Dropout2d(0.6)
        self.drop = nn.Dropout(0.6)


    def forward(self, x):
        x = self.conv1(x)
        x = self.drop2d(x) # dropout
        x = self.max_pool1(x)
        x = self.drop2d(x) # dropout
        x = F.relu(x)
        x = self.conv2(x)
        x = self.drop2d(x) # dropout
        x = self.max_pool2(x)
        x = self.drop2d(x) # dropout
        x = F.relu(x)
        x = self.drop(x.view(-1, 512)) # dropout
        x = self.fc1(x)
        x = F.relu(x)
        x = self.drop(x)
        x = self.fc2(x)
        return x

In [11]:
# return the accuracy between predicted and groundtruth, convert predicted values to {0, 1}
def accuracy(model, test_input, test_target):
    nb_samples = test_input.shape[0]
    model.train(False) # deactivate dropout
    output = model(test_input)
    model.train(True) # deactivate dropout
    output_int = torch.zeros(nb_samples)
    for i in range(nb_samples):
        if output[i][0] <= output[i][1]: # first digit lesser or equal
            output_int[i] = 1
    nb_errors = (output_int - test_target).type(torch.BoolTensor).sum().item()
    return (nb_samples - nb_errors) / nb_samples

In [ ]:
def weight_reset(m):
    if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
        m.reset_parameters()

In [12]:
# Train a model, print the results and return the train and validation loss.
def train_model(model, train_input, train_target, val_input, val_target, rounds, validation=True, verbose=False):
    
    model.apply(weight_reset) # reset weights
    
    batch_size, nb_epochs = 100, 200
    
    nb_epochs_shown = 5
    
    criterion = nn.CrossEntropyLoss()
    
    optimizer = optim.Adam(model.parameters(), lr = 1e-3)
    
    # tensors for averaging over the rounds
    times = torch.zeros(rounds)
    train_acc = torch.zeros(rounds)
    test_acc = torch.zeros(rounds)
    train_loss = torch.zeros(rounds, nb_epochs + 1)
    validation_loss = torch.zeros(rounds, nb_epochs + 1)
    
    for r in range(rounds):
            
        t0 = time.time()

        for e in range(nb_epochs):
            
            # store the train and validation loss for each epoch and round 
            if validation:
                model.train(False) # deactivate dropout
                train_loss[r,e] = criterion(model(train_input), train_target)
                validation_loss[r,e] = criterion(model(val_input), val_target)
                model.train(True) # deactivate dropout
            
            # updating the model
            for input, targets in zip(train_input.split(batch_size),train_target.split(batch_size)):
                output = model(input)
                loss = criterion(output, targets)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            # print the loss for a given number of epochs - used for direct feedback
            if verbose:
                if((e + 1) % int(nb_epochs / nb_epochs_shown) == 0):
                    print("Epoch {} | Loss : {}".format(e+1, loss))
                    
        # final loss            
        if validation:
            model.train(False) # deactivate dropout
            train_loss[r,nb_epochs] = criterion(model(train_input), train_target)
            validation_loss[r,nb_epochs] = criterion(model(val_input), val_target)
            model.train(True) # deactivate dropout

        t1 = time.time()
        
        times[r] = t1-t0
        train_acc[r] = accuracy(model, train_input, train_target)
        test_acc[r] = accuracy(model, test_input, test_target)
        
        print('Round {} done.'.format(r))
        
    print('--------------')
    
    total_trained_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print("Model : {} \n".format(model.__class__.__name__ ) + \
          "Number of trained parameters : {} \n".format(total_trained_params) + \
          "Size of mini-batches : {}\n".format(batch_size) + \
          "Averaged on {} rounds \n".format(rounds) + \
          "    Time for {} epochs : {:.2f}s\n".format(nb_epochs, times.mean().item()) + \
          "    Train accuracy : {:.3f} \n".format(train_acc.mean().item()) + \
          "    Test accuracy : {:.3f}".format(test_acc.mean().item()))
    
    return train_loss.detach().mean(dim=0), validation_loss.detach().mean(dim=0)

In [13]:
model = NetTest()
#model = NetTestDropout()
avg_train_loss, avg_val_loss = train_model(model, train_input, train_target, val_input, val_target, rounds=1)

ValueError: Expected input batch_size (40) to match target batch_size (10).